# Oil & Gas Field Operations — Agent Instructions
### Industry-Specific Instructions & Sample Queries for Data Agents

This notebook defines **detailed agent instructions** for the Oil & Gas industry.
Run it **before** `06_Create_Data_Agent.ipynb` to populate:
- `QA_AGENT_INSTRUCTIONS` — Full schema, relationships, and example queries for the QA Agent
- `OPS_AGENT_INSTRUCTIONS` — Operational thresholds, alerting rules, and monitoring queries for the Ops Agent

Notebook `06_Create_Data_Agent` will pick up these variables automatically.
If this notebook is not run, `06` falls back to generic instructions.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT INSTRUCTIONS — Oil & Gas Field Operations
# ════════════════════════════════════════════════════════════════════════

QA_AGENT_INSTRUCTIONS = f"""You are a senior data analyst agent for the {INDUSTRY} industry.
You answer ad-hoc questions about oil and gas field operations, drilling performance,
well integrity, HSE inspections, production performance, permit-to-work compliance,
and field engineer wellness using the connected data sources.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables, full historical data
   Dimension tables:
   - dim_equipment: equipment_id, name, type, manufacturer, model, install_date, last_service_date, status, well_site_id
   - dim_facilities: facility_id, name, type, region, capacity, operational_status, commissioning_date, operator
   - dim_field_engineers: engineer_id, name, role, department, hire_date, certifications, specialty, region
   - dim_regulatory_bodies: body_id, name, jurisdiction, inspection_frequency, penalty_range, focus_areas
   - dim_report_types: report_type_id, name, category, required_fields, frequency, regulatory_body_id
   - dim_well_sites: well_site_id, name, field, basin, latitude, longitude, well_type, spud_date, status, operator

   Batch fact tables:
   - fact_daily_drilling_reports: report_id, well_site_id, engineer_id, date, depth_ft, rop_ft_hr, mud_weight, incidents, npt_hours, status
   - fact_field_location: location_id, engineer_id, timestamp, lat, lon, well_site_id, activity_type
   - fact_field_wellness: survey_id, engineer_id, date, workload_score, overtime_hours, burnout_risk, fatigue_level, rotation_days
   - fact_hse_inspections: inspection_id, well_site_id, engineer_id, date, inspection_type, findings, severity, corrective_actions, compliance_flag
   - fact_operator_satisfaction: survey_id, facility_id, engineer_id, date, overall_score, safety_score, efficiency_score, communication_score, complaint_flag
   - fact_permit_to_work: permit_id, well_site_id, engineer_id, timestamp, work_type, risk_level, status, duration_hours, violations

   Event fact tables:
   - fact_production_alerts: alert_id, well_site_id, facility_id, timestamp, alert_type, severity, parameter, threshold, actual_value
   - fact_production_performance: perf_id, well_site_id, date, oil_bbl, gas_mcf, water_bbl, uptime_pct, efficiency_pct
   - fact_report_quality: quality_id, engineer_id, report_type_id, date, quality_score, completeness_pct, errors_found, revision_count
   - fact_scada_interactions: interaction_id, engineer_id, timestamp, equipment_id, action, duration_seconds, alarm_acknowledged, screen
   - fact_tour_handoffs: handoff_id, well_site_id, from_engineer_id, to_engineer_id, timestamp, doc_completeness_pct, pending_items, safety_briefing_flag
   - fact_well_integrity_events: event_id, well_site_id, engineer_id, timestamp, event_type, severity, pressure_psi, temperature_f, action_taken

2. SEMANTIC MODEL ({SEMANTIC_MODEL_NAME}) — DirectQuery over the Warehouse.
3. LAKEHOUSE ({LAKEHOUSE_NAME}) — Delta tables for Spark-based analysis.
4. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time event and streaming data (KQL).
   Event tables: production_alerts, production_performance, report_quality, scada_interactions, tour_handoffs, well_integrity_events
   Streaming tables: environmental_monitoring, hse_events, ptw_status, scada_alarms, well_telemetry

== KEY RELATIONSHIPS ==
- dim_field_engineers.engineer_id → fact tables on engineer_id (also from_engineer_id, to_engineer_id)
- dim_well_sites.well_site_id → fact tables on well_site_id
- dim_facilities.facility_id → fact_production_alerts, fact_operator_satisfaction
- dim_equipment.equipment_id → fact_scada_interactions
- dim_regulatory_bodies.body_id → dim_report_types
- dim_report_types.report_type_id → fact_report_quality

== EXAMPLE QUERIES ==

Q: What is the production performance by well site?
→ Query fact_production_performance, GROUP BY well_site_id, AVG(oil_bbl), AVG(uptime_pct), join dim_well_sites.

Q: Show drilling progress.
→ Query fact_daily_drilling_reports, GROUP BY well_site_id, MAX(depth_ft), AVG(rop_ft_hr), join dim_well_sites.

Q: Which well sites have integrity events?
→ Query fact_well_integrity_events, GROUP BY well_site_id, COUNT(*), join dim_well_sites.

Q: Show HSE inspection findings by severity.
→ Query fact_hse_inspections, GROUP BY severity, COUNT(*), SUM(corrective_actions).

Q: What is the permit-to-work compliance?
→ Query fact_permit_to_work, GROUP BY status, risk_level, COUNT(*).

Q: Show report quality by engineer.
→ Query fact_report_quality, GROUP BY engineer_id, AVG(quality_score), join dim_field_engineers.

Q: Show operator satisfaction by facility.
→ Query fact_operator_satisfaction, GROUP BY facility_id, AVG(overall_score), join dim_facilities.

Q: What are the tour handoff patterns?
→ Query fact_tour_handoffs, GROUP BY from_engineer_id, AVG(doc_completeness_pct), join dim_field_engineers.

Q: Show field engineer wellness.
→ Query fact_field_wellness, GROUP BY engineer_id, AVG(workload_score), join dim_field_engineers.

Q: What SCADA interactions are most common?
→ Query fact_scada_interactions, GROUP BY action, COUNT(*), AVG(duration_seconds).
"""

print(f"QA_AGENT_INSTRUCTIONS set ({len(QA_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT INSTRUCTIONS — Oil & Gas Field Operations
# ════════════════════════════════════════════════════════════════════════

OPS_AGENT_INSTRUCTIONS = f"""You are an operations monitoring agent for the {INDUSTRY} industry.
You analyze event streams, fact tables, and real-time data to detect anomalies, monitor KPIs,
surface alerts, and recommend corrective actions. Focus on well integrity, production performance,
HSE compliance, permit-to-work violations, equipment alarms, and field engineer safety.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables for historical analysis.
   Key operational tables:
   - fact_field_wellness: CRITICAL if burnout_risk = true, fatigue_level > 8
   - fact_well_integrity_events: CRITICAL if severity = 'Critical', pressure deviations
   - fact_production_alerts: CRITICAL if severity = 'Critical', actual_value exceeds threshold
   - fact_hse_inspections: severity = 'Critical', compliance_flag = false
   - fact_permit_to_work: violations > 0, risk_level = 'High'
   - fact_daily_drilling_reports: npt_hours > 4, incidents > 0
   - fact_production_performance: uptime_pct < 90, efficiency_pct < 80
   - fact_operator_satisfaction: overall_score < 5, complaint_flag = true
   - fact_tour_handoffs: doc_completeness_pct < 80, safety_briefing_flag = false
   - fact_report_quality: quality_score < 70, errors_found > 5
   - fact_scada_interactions: alarm_acknowledged = false

   Dimension tables: dim_field_engineers, dim_well_sites, dim_facilities, dim_equipment, dim_regulatory_bodies, dim_report_types

2. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time and streaming data.
   Streaming tables:
   - well_telemetry: reading_id, well_site_id, timestamp, pressure_psi, temperature_f, flow_rate, casing_pressure
     → CRITICAL: pressure_psi deviation > 20% from normal, casing_pressure anomaly
   - scada_alarms: alarm_id, equipment_id, timestamp, alarm_type, severity, acknowledged, well_site_id
     → CRITICAL: severity = 'Critical', acknowledged = false
   - hse_events: event_id, well_site_id, timestamp, event_type, severity, personnel_involved, injuries
     → CRITICAL: injuries > 0
   - environmental_monitoring: reading_id, well_site_id, timestamp, parameter, value, threshold, compliance_flag
     → Alert on compliance_flag = false
   - ptw_status: permit_id, well_site_id, timestamp, work_type, status, risk_level, engineer_id
     → Alert on violations

== ALERTING THRESHOLDS ==
- Well Integrity: severity = 'Critical', pressure deviation > 20%
- Production: uptime_pct < 90, efficiency_pct < 80
- HSE: severity = 'Critical', compliance = false, injuries > 0
- Equipment: unacknowledged critical SCADA alarms
- Personnel: burnout_risk = true, fatigue_level > 8
- PTW: violations > 0, risk_level = 'High'
- Drilling: npt_hours > 4, incidents > 0
- Environment: compliance_flag = false

== EXAMPLE QUERIES ==

Q: Are there critical well integrity events?
→ Query fact_well_integrity_events WHERE severity = 'Critical', join dim_well_sites.

Q: Which wells have production issues?
→ Query fact_production_performance WHERE uptime_pct < 90, join dim_well_sites.

Q: Show HSE inspection failures.
→ Query fact_hse_inspections WHERE compliance_flag = false, join dim_well_sites.

Q: Show real-time SCADA alarms.
→ KQL: scada_alarms | where severity == 'Critical' and acknowledged == false | project equipment_id, alarm_type, well_site_id

Q: Are there environmental compliance issues?
→ KQL: environmental_monitoring | where compliance_flag == false | project well_site_id, parameter, value, threshold

Q: Show well telemetry anomalies.
→ KQL: well_telemetry | where timestamp > ago(1h) | summarize avg(pressure_psi) by well_site_id

Q: Which permit-to-work entries have violations?
→ Query fact_permit_to_work WHERE violations > 0, join dim_well_sites.

Q: Which field engineers are fatigued?
→ Query fact_field_wellness WHERE burnout_risk = true, join dim_field_engineers.

Q: Show drilling incidents.
→ Query fact_daily_drilling_reports WHERE incidents > 0, join dim_well_sites.

Q: Show unacknowledged SCADA alarms.
→ KQL: scada_alarms | where acknowledged == false | summarize count() by alarm_type, severity
"""

print(f"OPS_AGENT_INSTRUCTIONS set ({len(OPS_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PER-DATASOURCE INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════

QA_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse contains all oil & gas field operations data as SQL tables.

DIMENSION TABLES:
- dim_equipment, dim_facilities, dim_field_engineers, dim_regulatory_bodies, dim_report_types, dim_well_sites

FACT TABLES:
- fact_daily_drilling_reports, fact_field_location, fact_field_wellness, fact_hse_inspections,
  fact_operator_satisfaction, fact_permit_to_work

EVENT FACT TABLES:
- fact_production_alerts, fact_production_performance, fact_report_quality, fact_scada_interactions,
  fact_tour_handoffs, fact_well_integrity_events

KEY JOINS:
- dim_field_engineers.engineer_id → fact tables on engineer_id
- dim_well_sites.well_site_id → fact tables on well_site_id
- dim_facilities.facility_id → fact_production_alerts, fact_operator_satisfaction
- dim_equipment.equipment_id → fact_scada_interactions""",

    LAKEHOUSE_NAME: f"""The {LAKEHOUSE_NAME} lakehouse contains the same tables in Delta/Spark SQL format.
Same schema and joins as the Warehouse. Use LIMIT instead of TOP for row limits.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database contains real-time event and streaming tables.

EVENT TABLES (KQL):
- production_alerts, production_performance, report_quality, scada_interactions, tour_handoffs, well_integrity_events

STREAMING TABLES:
- environmental_monitoring, hse_events, ptw_status, scada_alarms, well_telemetry

Use KQL syntax (not SQL).""",
}

OPS_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse is the primary data source for operational monitoring.

KEY OPERATIONAL TABLES AND THRESHOLDS:
- fact_well_integrity_events: CRITICAL on severity = 'Critical'
- fact_production_performance: uptime_pct < 90, efficiency_pct < 80
- fact_hse_inspections: severity = 'Critical', compliance_flag = false
- fact_field_wellness: burnout_risk = true, fatigue_level > 8
- fact_permit_to_work: violations > 0, risk_level = 'High'
- fact_daily_drilling_reports: npt_hours > 4, incidents > 0

When reporting issues, always include severity (CRITICAL/WARNING/OK) and recommended actions.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database provides real-time telemetry.

STREAMING ALERTS:
- well_telemetry: pressure deviation > 20%
- scada_alarms: severity = 'Critical', acknowledged = false
- hse_events: injuries > 0
- environmental_monitoring: compliance_flag = false

Use KQL syntax. Prefer summarize/count/avg for aggregations. Use ago(24h) for recent activity.""",
}

print(f"QA_DS_INSTRUCTIONS set: {', '.join(QA_DS_INSTRUCTIONS.keys())}")
print(f"OPS_DS_INSTRUCTIONS set: {', '.join(OPS_DS_INSTRUCTIONS.keys())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

QA_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "What is the production performance by well site?",
            "query": """SELECT ws.name AS well_site, ws.field, ws.basin,\n       AVG(pp.oil_bbl) AS avg_oil_bbl,\n       AVG(pp.gas_mcf) AS avg_gas_mcf,\n       AVG(pp.uptime_pct) AS avg_uptime,\n       AVG(pp.efficiency_pct) AS avg_efficiency\nFROM fact_production_performance pp\nJOIN dim_well_sites ws ON pp.well_site_id = ws.well_site_id\nGROUP BY ws.name, ws.field, ws.basin\nORDER BY avg_oil_bbl DESC"""
        },
        {
            "question": "Show drilling progress.",
            "query": """SELECT ws.name AS well_site,\n       MAX(ddr.depth_ft) AS max_depth,\n       AVG(ddr.rop_ft_hr) AS avg_rop,\n       SUM(ddr.npt_hours) AS total_npt,\n       SUM(ddr.incidents) AS total_incidents\nFROM fact_daily_drilling_reports ddr\nJOIN dim_well_sites ws ON ddr.well_site_id = ws.well_site_id\nGROUP BY ws.name\nORDER BY max_depth DESC"""
        },
        {
            "question": "Which well sites have integrity events?",
            "query": """SELECT ws.name AS well_site, wie.event_type, wie.severity,\n       COUNT(*) AS events,\n       AVG(wie.pressure_psi) AS avg_pressure\nFROM fact_well_integrity_events wie\nJOIN dim_well_sites ws ON wie.well_site_id = ws.well_site_id\nGROUP BY ws.name, wie.event_type, wie.severity\nORDER BY events DESC"""
        },
        {
            "question": "Show HSE inspection findings.",
            "query": """SELECT hi.severity, hi.inspection_type,\n       COUNT(*) AS inspections,\n       SUM(CASE WHEN hi.compliance_flag = false THEN 1 ELSE 0 END) AS non_compliant\nFROM fact_hse_inspections hi\nGROUP BY hi.severity, hi.inspection_type\nORDER BY non_compliant DESC"""
        },
        {
            "question": "Show permit-to-work status.",
            "query": """SELECT ptw.status, ptw.risk_level, ptw.work_type,\n       COUNT(*) AS permits,\n       SUM(ptw.violations) AS total_violations\nFROM fact_permit_to_work ptw\nGROUP BY ptw.status, ptw.risk_level, ptw.work_type\nORDER BY total_violations DESC"""
        },
        {
            "question": "Show report quality by engineer.",
            "query": """SELECT fe.name, fe.role,\n       AVG(rq.quality_score) AS avg_quality,\n       AVG(rq.completeness_pct) AS avg_completeness,\n       SUM(rq.errors_found) AS total_errors\nFROM fact_report_quality rq\nJOIN dim_field_engineers fe ON rq.engineer_id = fe.engineer_id\nGROUP BY fe.name, fe.role\nORDER BY avg_quality DESC"""
        },
        {
            "question": "Show operator satisfaction by facility.",
            "query": """SELECT f.name AS facility, f.type,\n       AVG(os.overall_score) AS avg_overall,\n       AVG(os.safety_score) AS avg_safety,\n       AVG(os.efficiency_score) AS avg_efficiency,\n       SUM(CASE WHEN os.complaint_flag = true THEN 1 ELSE 0 END) AS complaints\nFROM fact_operator_satisfaction os\nJOIN dim_facilities f ON os.facility_id = f.facility_id\nGROUP BY f.name, f.type\nORDER BY avg_overall DESC"""
        },
        {
            "question": "Show tour handoff completeness.",
            "query": """SELECT fe.name AS from_engineer,\n       COUNT(*) AS handoffs,\n       AVG(th.doc_completeness_pct) AS avg_completeness,\n       SUM(CASE WHEN th.safety_briefing_flag = false THEN 1 ELSE 0 END) AS no_briefing\nFROM fact_tour_handoffs th\nJOIN dim_field_engineers fe ON th.from_engineer_id = fe.engineer_id\nGROUP BY fe.name\nORDER BY avg_completeness ASC"""
        },
        {
            "question": "Show field engineer wellness.",
            "query": """SELECT fe.name, fe.role,\n       fw.workload_score, fw.overtime_hours,\n       fw.fatigue_level, fw.rotation_days, fw.burnout_risk\nFROM fact_field_wellness fw\nJOIN dim_field_engineers fe ON fw.engineer_id = fe.engineer_id\nORDER BY fw.workload_score DESC"""
        },
        {
            "question": "Show SCADA interactions by action.",
            "query": """SELECT si.action, si.screen,\n       COUNT(*) AS interactions,\n       AVG(si.duration_seconds) AS avg_duration,\n       SUM(CASE WHEN si.alarm_acknowledged = false THEN 1 ELSE 0 END) AS unacknowledged\nFROM fact_scada_interactions si\nGROUP BY si.action, si.screen\nORDER BY interactions DESC"""
        },
    ],

    LAKEHOUSE_NAME: [
        {
            "question": "Which wells have the highest production?",
            "query": """SELECT ws.name, ws.field,\n       SUM(pp.oil_bbl) AS total_oil,\n       SUM(pp.gas_mcf) AS total_gas\nFROM fact_production_performance pp\nJOIN dim_well_sites ws ON pp.well_site_id = ws.well_site_id\nGROUP BY ws.name, ws.field\nORDER BY total_oil DESC\nLIMIT 20"""
        },
        {
            "question": "Show well sites by basin.",
            "query": """SELECT basin, well_type, status,\n       COUNT(*) AS wells\nFROM dim_well_sites\nGROUP BY basin, well_type, status\nORDER BY wells DESC"""
        },
        {
            "question": "Show equipment status.",
            "query": """SELECT type, status, manufacturer,\n       COUNT(*) AS count\nFROM dim_equipment\nGROUP BY type, status, manufacturer\nORDER BY count DESC"""
        },
        {
            "question": "Show production alerts by type.",
            "query": """SELECT alert_type, severity,\n       COUNT(*) AS alerts\nFROM fact_production_alerts\nGROUP BY alert_type, severity\nORDER BY alerts DESC"""
        },
        {
            "question": "Show field engineer certifications.",
            "query": """SELECT specialty, region, COUNT(*) AS engineers\nFROM dim_field_engineers\nGROUP BY specialty, region\nORDER BY engineers DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show well telemetry readings.",
            "query": """well_telemetry\n| where timestamp > ago(1h)\n| summarize avg_pressure = avg(pressure_psi), avg_temp = avg(temperature_f), avg_flow = avg(flow_rate) by well_site_id\n| order by avg_pressure desc"""
        },
        {
            "question": "Show SCADA alarms.",
            "query": """scada_alarms\n| where timestamp > ago(24h)\n| summarize count() by alarm_type, severity, acknowledged\n| order by count_ desc"""
        },
        {
            "question": "Show HSE events.",
            "query": """hse_events\n| where timestamp > ago(24h)\n| summarize count(), total_injuries = sum(injuries) by event_type, severity\n| order by total_injuries desc"""
        },
        {
            "question": "Show environmental monitoring.",
            "query": """environmental_monitoring\n| where timestamp > ago(24h)\n| where compliance_flag == false\n| project well_site_id, parameter, value, threshold\n| order by timestamp desc"""
        },
        {
            "question": "Show permit-to-work status.",
            "query": """ptw_status\n| where timestamp > ago(24h)\n| summarize count() by work_type, status, risk_level\n| order by count_ desc"""
        },
    ],
}

print(f"QA_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in QA_FEWSHOTS.items())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

OPS_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Are there critical well integrity events?",
            "query": """SELECT ws.name AS well_site, wie.event_type,\n       wie.severity, wie.pressure_psi, wie.temperature_f,\n       wie.action_taken, fe.name AS engineer\nFROM fact_well_integrity_events wie\nJOIN dim_well_sites ws ON wie.well_site_id = ws.well_site_id\nJOIN dim_field_engineers fe ON wie.engineer_id = fe.engineer_id\nWHERE wie.severity = 'Critical'\nORDER BY wie.timestamp DESC"""
        },
        {
            "question": "Which wells have production issues?",
            "query": """SELECT ws.name AS well_site, ws.field,\n       pp.uptime_pct, pp.efficiency_pct,\n       pp.oil_bbl, pp.gas_mcf,\n       CASE WHEN pp.uptime_pct < 80 THEN 'CRITICAL'\n            WHEN pp.uptime_pct < 90 THEN 'WARNING'\n            ELSE 'OK' END AS severity\nFROM fact_production_performance pp\nJOIN dim_well_sites ws ON pp.well_site_id = ws.well_site_id\nWHERE pp.uptime_pct < 90\nORDER BY pp.uptime_pct ASC"""
        },
        {
            "question": "Show HSE inspection failures.",
            "query": """SELECT ws.name AS well_site, hi.inspection_type,\n       hi.severity, hi.findings, hi.corrective_actions\nFROM fact_hse_inspections hi\nJOIN dim_well_sites ws ON hi.well_site_id = ws.well_site_id\nWHERE hi.compliance_flag = false\nORDER BY hi.date DESC"""
        },
        {
            "question": "Which permit-to-work entries have violations?",
            "query": """SELECT ws.name AS well_site, ptw.work_type,\n       ptw.risk_level, ptw.violations, ptw.status,\n       fe.name AS engineer\nFROM fact_permit_to_work ptw\nJOIN dim_well_sites ws ON ptw.well_site_id = ws.well_site_id\nJOIN dim_field_engineers fe ON ptw.engineer_id = fe.engineer_id\nWHERE ptw.violations > 0\nORDER BY ptw.violations DESC"""
        },
        {
            "question": "Which field engineers are fatigued?",
            "query": """SELECT fe.name, fe.role, fe.region,\n       fw.workload_score, fw.fatigue_level,\n       fw.overtime_hours, fw.rotation_days, fw.burnout_risk\nFROM fact_field_wellness fw\nJOIN dim_field_engineers fe ON fw.engineer_id = fe.engineer_id\nWHERE fw.burnout_risk = true\nORDER BY fw.fatigue_level DESC"""
        },
        {
            "question": "Show drilling incidents.",
            "query": """SELECT ws.name AS well_site, ddr.date,\n       ddr.incidents, ddr.npt_hours, ddr.depth_ft, ddr.status\nFROM fact_daily_drilling_reports ddr\nJOIN dim_well_sites ws ON ddr.well_site_id = ws.well_site_id\nWHERE ddr.incidents > 0\nORDER BY ddr.incidents DESC"""
        },
        {
            "question": "Show production alerts.",
            "query": """SELECT ws.name AS well_site, pa.alert_type,\n       pa.severity, pa.parameter, pa.threshold, pa.actual_value\nFROM fact_production_alerts pa\nJOIN dim_well_sites ws ON pa.well_site_id = ws.well_site_id\nWHERE pa.severity = 'Critical'\nORDER BY pa.timestamp DESC"""
        },
        {
            "question": "Show tour handoffs missing safety briefings.",
            "query": """SELECT ws.name AS well_site,\n       fe.name AS from_engineer,\n       th.doc_completeness_pct, th.pending_items\nFROM fact_tour_handoffs th\nJOIN dim_well_sites ws ON th.well_site_id = ws.well_site_id\nJOIN dim_field_engineers fe ON th.from_engineer_id = fe.engineer_id\nWHERE th.safety_briefing_flag = false\nORDER BY th.doc_completeness_pct ASC"""
        },
        {
            "question": "Show operator complaints.",
            "query": """SELECT f.name AS facility,\n       os.overall_score, os.safety_score, os.efficiency_score\nFROM fact_operator_satisfaction os\nJOIN dim_facilities f ON os.facility_id = f.facility_id\nWHERE os.complaint_flag = true\nORDER BY os.overall_score ASC"""
        },
        {
            "question": "Show unacknowledged SCADA interactions.",
            "query": """SELECT e.name AS equipment, si.action,\n       si.duration_seconds, fe.name AS engineer\nFROM fact_scada_interactions si\nJOIN dim_equipment e ON si.equipment_id = e.equipment_id\nJOIN dim_field_engineers fe ON si.engineer_id = fe.engineer_id\nWHERE si.alarm_acknowledged = false\nORDER BY si.timestamp DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show critical SCADA alarms.",
            "query": """scada_alarms\n| where severity == 'Critical' and acknowledged == false\n| project equipment_id, alarm_type, well_site_id\n| order by timestamp desc"""
        },
        {
            "question": "Show well telemetry anomalies.",
            "query": """well_telemetry\n| where timestamp > ago(1h)\n| summarize avg_pressure = avg(pressure_psi), avg_casing = avg(casing_pressure) by well_site_id\n| order by avg_pressure desc"""
        },
        {
            "question": "Are there HSE injuries?",
            "query": """hse_events\n| where injuries > 0\n| project well_site_id, event_type, severity, personnel_involved, injuries\n| order by injuries desc"""
        },
        {
            "question": "Show environmental compliance issues.",
            "query": """environmental_monitoring\n| where compliance_flag == false\n| project well_site_id, parameter, value, threshold\n| order by timestamp desc"""
        },
        {
            "question": "Show permit-to-work status.",
            "query": """ptw_status\n| where timestamp > ago(24h)\n| summarize count() by work_type, status, risk_level\n| order by count_ desc"""
        },
        {
            "question": "Show SCADA alarm trend.",
            "query": """scada_alarms\n| where timestamp > ago(7d)\n| summarize count() by bin(timestamp, 1d), severity\n| order by timestamp asc"""
        },
        {
            "question": "Show well telemetry pressure trend.",
            "query": """well_telemetry\n| where timestamp > ago(24h)\n| summarize avg_pressure = avg(pressure_psi) by bin(timestamp, 1h), well_site_id\n| order by timestamp asc"""
        },
    ],
}

print(f"OPS_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in OPS_FEWSHOTS.items())}")

## Sample Questions for the Oil & Gas Data Agents

### QA Agent — `OilAndGas_QA_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Production | What is the production performance by well site? |
| 2 | Drilling | Show drilling progress. |
| 3 | Integrity | Which well sites have integrity events? |
| 4 | HSE | Show HSE inspection findings. |
| 5 | PTW | Show permit-to-work status. |
| 6 | Quality | Show report quality by engineer. |
| 7 | Satisfaction | Show operator satisfaction by facility. |
| 8 | Handoffs | Show tour handoff completeness. |
| 9 | Wellness | Show field engineer wellness. |
| 10 | SCADA | Show SCADA interactions by action. |
| 11 | Wells | Which wells have the highest production? |
| 12 | Equipment | Show equipment status. |
| 13 | Telemetry | Show well telemetry readings. |
| 14 | Alarms | Show SCADA alarms. |
| 15 | Environment | Show environmental monitoring. |

### Ops Agent — `OilAndGas_Ops_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Integrity | Are there critical well integrity events? |
| 2 | Production | Which wells have production issues? |
| 3 | HSE | Show HSE inspection failures. |
| 4 | PTW | Which permit-to-work entries have violations? |
| 5 | Wellness | Which field engineers are fatigued? |
| 6 | Drilling | Show drilling incidents. |
| 7 | Alerts | Show production alerts. |
| 8 | Handoffs | Show tour handoffs missing safety briefings. |
| 9 | SCADA | Show critical SCADA alarms. |
| 10 | Telemetry | Show well telemetry anomalies. |
| 11 | HSE | Are there HSE injuries? |
| 12 | Environment | Show environmental compliance issues. |
| 13 | Satisfaction | Show operator complaints. |
| 14 | Trends | Show SCADA alarm trend. |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SAVE INSTRUCTIONS FOR PARENT NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

import json

_result = json.dumps({
    "qa": QA_AGENT_INSTRUCTIONS,
    "ops": OPS_AGENT_INSTRUCTIONS,
    "qa_fewshots": QA_FEWSHOTS,
    "ops_fewshots": OPS_FEWSHOTS,
    "qa_ds_instructions": QA_DS_INSTRUCTIONS,
    "ops_ds_instructions": OPS_DS_INSTRUCTIONS
})

_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
notebookutils.fs.put(_tmp_path, _result, True)
print(f"  Written {len(_result):,} bytes to {_tmp_path}")

print(f"{'═'*60}")
print(f"AGENT INSTRUCTIONS LOADED — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  QA_AGENT_INSTRUCTIONS:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
print(f"  OPS_AGENT_INSTRUCTIONS: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
print(f"  QA_FEWSHOTS:  {sum(len(v) for v in QA_FEWSHOTS.values())} total queries across {len(QA_FEWSHOTS)} datasources")
print(f"  OPS_FEWSHOTS: {sum(len(v) for v in OPS_FEWSHOTS.values())} total queries across {len(OPS_FEWSHOTS)} datasources")

notebookutils.notebook.exit("ok")